# 1. Setup & Load Data

In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [4]:
df = pd.read_csv(
    '../data/processed/online_retail_cleaned.csv'
)

C:\Users\user\AppData\Local\Temp\ipykernel_400\2852433281.py:1: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [5]:
df['invoice_no_normalized'] = (
    df['InvoiceNo']
      .astype(str)
      .str.strip()
)

In [6]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [7]:
print("Rows :", f"{len(df):,}")
print("Columns :", df.shape[1])

print(
    "Unique normalized invoices :",
    f"{df['invoice_no_normalized'].nunique():,}"
)

print(
    "Unique customers :",
    f"{df['CustomerID'].nunique():,}"
)

display(df.head())

Rows : 524,878
Columns : 11
Unique normalized invoices : 19,960
Unique customers : 4,338


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancelled,Revenue,invoice_no_normalized
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom,False,15.30,536365
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom,False,20.34,536365
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom,False,22.00,536365
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom,False,20.34,536365
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom,False,20.34,536365


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 524878 entries, 0 to 524877
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   InvoiceNo              524878 non-null  object        
 1   StockCode              524878 non-null  str           
 2   Description            524878 non-null  str           
 3   Quantity               524878 non-null  int64         
 4   InvoiceDate            524878 non-null  datetime64[us]
 5   UnitPrice              524878 non-null  float64       
 6   CustomerID             392692 non-null  float64       
 7   Country                524878 non-null  str           
 8   is_cancelled           524878 non-null  bool          
 9   Revenue                524878 non-null  float64       
 10  invoice_no_normalized  524878 non-null  str           
dtypes: bool(1), datetime64[us](1), float64(3), int64(1), object(1), str(4)
memory usage: 40.5+ MB


# 2. Define Analysis Date

In [9]:
latest_transaction_date = df['InvoiceDate'].max()

print(
    "Latest transaction date :",
    latest_transaction_date
)

Latest transaction date : 2011-12-09 12:50:00


In [10]:
analysis_date = latest_transaction_date + pd.Timedelta(days=1)

print(
    "Analysis date :",
    analysis_date
)

Analysis date : 2011-12-10 12:50:00


# 3. Calculate Recency

In [11]:
customer_last_purchase = (
    df[df['CustomerID'].notna()]
    .groupby('CustomerID', as_index=False)
    .agg(
        last_purchase_date=('InvoiceDate', 'max')
    )
)

customer_last_purchase.head(10)

,CustomerID,last_purchase_date
0,"12,346.00",2011-01-18 10:01:00
1,"12,347.00",2011-12-07 15:52:00
2,"12,348.00",2011-09-25 13:13:00
3,"12,349.00",2011-11-21 09:51:00
4,"12,350.00",2011-02-02 16:01:00
5,"12,352.00",2011-11-03 14:37:00
6,"12,353.00",2011-05-19 17:47:00
7,"12,354.00",2011-04-21 13:11:00
8,"12,355.00",2011-05-09 13:49:00
9,"12,356.00",2011-11-17 08:40:00


In [12]:
customer_last_purchase['recency'] = (
    analysis_date - customer_last_purchase['last_purchase_date']
).dt.days

In [13]:
customer_last_purchase.head(10)

,CustomerID,last_purchase_date,recency
0,"12,346.00",2011-01-18 10:01:00,326
1,"12,347.00",2011-12-07 15:52:00,2
2,"12,348.00",2011-09-25 13:13:00,75
3,"12,349.00",2011-11-21 09:51:00,19
4,"12,350.00",2011-02-02 16:01:00,310
5,"12,352.00",2011-11-03 14:37:00,36
6,"12,353.00",2011-05-19 17:47:00,204
7,"12,354.00",2011-04-21 13:11:00,232
8,"12,355.00",2011-05-09 13:49:00,214
9,"12,356.00",2011-11-17 08:40:00,23


In [14]:
customer_last_purchase['recency'].describe()

count   4,338.00
mean       92.54
std       100.01
min         1.00
25%        18.00
50%        51.00
75%       142.00
max       374.00
Name: recency, dtype: float64

# 4. Calculate Frequency

In [15]:
customer_frequency = (
    df[df['CustomerID'].notna()]
    .groupby('CustomerID', as_index=False)
    .agg(
        frequency=('invoice_no_normalized', 'nunique')
    )
)

customer_frequency.head(10)

,CustomerID,frequency
0,"12,346.00",1
1,"12,347.00",7
2,"12,348.00",4
3,"12,349.00",1
4,"12,350.00",1
5,"12,352.00",8
6,"12,353.00",1
7,"12,354.00",1
8,"12,355.00",1
9,"12,356.00",3


In [16]:
customer_frequency['frequency'].describe()

count   4,338.00
mean        4.27
std         7.70
min         1.00
25%         1.00
50%         2.00
75%         5.00
max       209.00
Name: frequency, dtype: float64

# 5. Calculate Monetary

In [17]:
customer_monetary = (
    df[df['CustomerID'].notna()]
    .groupby('CustomerID', as_index=False)
    .agg(
        monetary=('Revenue', 'sum')
    )
)

customer_monetary.head(10)

,CustomerID,monetary
0,"12,346.00","77,183.60"
1,"12,347.00","4,310.00"
2,"12,348.00","1,797.24"
3,"12,349.00","1,757.55"
4,"12,350.00",334.40
5,"12,352.00","2,506.04"
6,"12,353.00",89.00
7,"12,354.00","1,079.40"
8,"12,355.00",459.40
9,"12,356.00","2,811.43"


In [18]:
customer_monetary['monetary'].describe()

count     4,338.00
mean      2,048.69
std       8,985.23
min           3.75
25%         306.48
50%         668.57
75%       1,660.60
max     280,206.02
Name: monetary, dtype: float64

# 6. RFM Scoring

In [19]:
rfm = (
    customer_last_purchase[
        ['CustomerID', 'recency']
    ]
    .merge(
        customer_frequency[
            ['CustomerID', 'frequency']
        ],
        on='CustomerID',
        how='inner'
    )
    .merge(
        customer_monetary[
            ['CustomerID', 'monetary']
        ],
        on='CustomerID',
        how='inner'
    )
)

rfm.head(10)

,CustomerID,recency,frequency,monetary
0,"12,346.00",326,1,"77,183.60"
1,"12,347.00",2,7,"4,310.00"
2,"12,348.00",75,4,"1,797.24"
3,"12,349.00",19,1,"1,757.55"
4,"12,350.00",310,1,334.40
5,"12,352.00",36,8,"2,506.04"
6,"12,353.00",204,1,89.00
7,"12,354.00",232,1,"1,079.40"
8,"12,355.00",214,1,459.40
9,"12,356.00",23,3,"2,811.43"


In [20]:
print("RFM rows :", f"{len(rfm):,}")
print("RFM columns :", rfm.shape[1])

RFM rows : 4,338
RFM columns : 4


In [21]:
print(
    "Expected customers :",
    f"{df['CustomerID'].nunique():,}"
)

print(
    "RFM customers      :",
    f"{rfm['CustomerID'].nunique():,}"
)

print(
    "Customer count match :",
    rfm['CustomerID'].nunique()
    == df['CustomerID'].nunique()
)

Expected customers : 4,338
RFM customers      : 4,338
Customer count match : True


In [22]:
print(
    "Duplicate customers :",
    rfm['CustomerID'].duplicated().sum()
)

Duplicate customers : 0


In [23]:
print(
    rfm[
        ['recency', 'frequency', 'monetary']
    ].isna().sum()
)

recency      0
frequency    0
monetary     0
dtype: int64


## 6.1 RFM Score - Recency

In [24]:
rfm['R_score'] = pd.qcut(
    rfm['recency'],
    q=5,
    labels=[5,4,3,2,1],
    duplicates='drop'
).astype(int)

In [25]:
rfm['R_score'].value_counts().sort_index()

R_score
1    865
2    843
3    858
4    904
5    868
Name: count, dtype: int64

In [26]:
print(
    rfm.groupby('R_score')['recency']
    .agg(['min', 'max', 'mean', 'count'])
    .sort_index(ascending=False)
)

         min  max   mean  count
R_score                        
5          1   13   6.17    868
4         14   33  23.06    904
3         34   72  52.32    858
2         73  179 116.25    843
1        180  374 268.60    865


## 6.2 RFM Score — Frequency

In [27]:
rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [28]:
rfm['F_score'].value_counts().sort_index()

F_score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64

In [29]:
f_score_means = (
    rfm.groupby('F_score')['frequency']
    .mean()
)

print(f_score_means)

F_score
1    1.00
2    1.28
3    2.32
4    4.02
5   12.74
Name: frequency, dtype: float64


## 6.3 RFM Score - Monetary

In [30]:
rfm['M_score'] = pd.qcut(
    rfm['monetary'],
    q=5,
    labels=[1,2,3,4,5],
    duplicates='drop'
).astype(int)

In [31]:
rfm['M_score'].value_counts().sort_index()

M_score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64

In [32]:
m_score_means = (
    rfm.groupby('M_score')['monetary']
    .mean()
)

print(m_score_means)

M_score
1     152.58
2     357.56
3     684.34
4   1,399.61
5   7,646.66
Name: monetary, dtype: float64


## 6.4 Calculate RFM Score

In [33]:
rfm['RFM_score'] = (
    rfm['R_score']
    + rfm['F_score']
    + rfm['M_score']
)

In [34]:
rfm['RFM_score'].value_counts().sort_index()

RFM_score
3     183
4     361
5     337
6     426
7     377
8     375
9     336
10    342
11    347
12    321
13    286
14    300
15    347
Name: count, dtype: int64

In [35]:
rfm[
    [
        'CustomerID',
        'recency',
        'frequency',
        'monetary',
        'R_score',
        'F_score',
        'M_score',
        'RFM_score'
    ]
].sort_values(
    'RFM_score',
    ascending=False
).head(20)

,CustomerID,recency,frequency,monetary,R_score,F_score,M_score,RFM_score
4309,"18,245.00",7,7,"2,567.06",5,5,5,15
4307,"18,241.00",10,17,"2,073.09",5,5,5,15
4298,"18,230.00",9,7,"2,810.20",5,5,5,15
4297,"18,229.00",12,20,"7,276.90",5,5,5,15
4293,"18,225.00",3,12,"5,504.96",5,5,5,15
4291,"18,223.00",5,14,"6,484.54",5,5,5,15
4287,"18,219.00",3,10,"2,069.77",5,5,5,15
4279,"18,210.00",2,6,"2,621.38",5,5,5,15
4272,"18,198.00",4,17,"5,425.56",5,5,5,15
4231,"18,144.00",8,12,"2,888.75",5,5,5,15


# 7. Segmentation

## 7.1 Create Segments

### 7.1.1 Business Rules

| Segment                        | Business Rule           | Interpretasi                                                                                  |
| ------------------------------ | ----------------------- | --------------------------------------------------------------------------------------------- |
| **Champions**                  | **R ≥ 4, F ≥ 4, M ≥ 4** | Best customers: recent, frequent, and high-value                                            |
| **Low Value Loyalists**        | **F ≥ 4, M ≤ 2**        | Frequent purchasers but with relatively low monetary value; potential for upselling/cross-selling |
| **Loyal Customers**            | **F ≥ 4, M ≥ 3**        | Customers with high purchase frequency and good monetary value                        |
| **Potential Loyalists**        | **R ≥ 4, F ≤ 3**        | Customers who are relatively recent but have low purchase frequency; they have the potential to become loyal          |
| **At Risk**                    | **R ≤ 2, F ≥ 2**        | Customers who were previously quite active but are no longer recent                                |
| **Lost Customers**             | **R ≤ 2, F = 1**        | Customers with low activity and no recent engagement                                       |
| **New / Developing Customers** | **R ≥ 3, F ≤ 3**        | A relatively recent customer whose purchasing activity is still growing                   |

## 7.1.2 Create Segment

In [36]:
def assign_segment(row):
    r = row['R_score']
    f = row['F_score']
    m = row['M_score']

    # 1. Best customers
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'

    # 2. Frequent customers with relatively low monetary value
    elif f >= 4 and m <= 2:
        return 'Low Value Loyalists'

    # 3. Frequent and valuable customers
    elif f >= 4 and m >= 3:
        return 'Loyal Customers'

    # 4. Recent customers with growth potential
    elif r >= 4 and f <= 3:
        return 'Potential Loyalists'

    # 5. Previously active but becoming inactive
    elif r <= 2 and f >= 2:
        return 'At Risk'

    # 6. Low activity and not recent
    elif r <= 2 and f == 1:
        return 'Lost Customers'

    # 7. Recent / developing customers
    elif r >= 3 and f <= 3:
        return 'New / Developing Customers'

    # Fallback
    else:
        return 'Other'

In [37]:
rfm['segment'] = rfm.apply(assign_segment, axis=1)

In [38]:
rfm['segment'].value_counts()

segment
Champions                     957
At Risk                       870
Loyal Customers               693
Potential Loyalists           633
Lost Customers                563
New / Developing Customers    537
Low Value Loyalists            85
Name: count, dtype: int64

In [39]:
rfm['segment'].value_counts(normalize=True).mul(100).round(2)

segment
Champions                    22.06
At Risk                      20.06
Loyal Customers              15.98
Potential Loyalists          14.59
Lost Customers               12.98
New / Developing Customers   12.38
Low Value Loyalists           1.96
Name: proportion, dtype: float64

In [40]:
print(
    "Customers in Other :",
    (rfm['segment'] == 'Other').sum()
)

Customers in Other : 0


In [41]:
rfm[
    [
        'CustomerID',
        'recency',
        'frequency',
        'monetary',
        'R_score',
        'F_score',
        'M_score',
        'RFM_score',
        'segment'
    ]
].sort_values(
    ['segment', 'RFM_score'],
    ascending=[True, False]
).head(30)

,CustomerID,recency,frequency,monetary,R_score,F_score,M_score,RFM_score,segment
50,"12,409.00",79,3,"11,072.67",2,3,5,10,At Risk
263,"12,669.00",151,3,"2,744.03",2,3,5,10,At Risk
589,"13,124.00",90,3,"3,756.33",2,3,5,10,At Risk
744,"13,334.00",82,3,"3,502.32",2,3,5,10,At Risk
1018,"13,722.00",132,3,"2,375.41",2,3,5,10,At Risk
1073,"13,802.00",139,3,"4,599.42",2,3,5,10,At Risk
1111,"13,851.00",96,3,"2,651.46",2,3,5,10,At Risk
1550,"14,461.00",148,3,"2,103.06",2,3,5,10,At Risk
1890,"14,930.00",109,3,"2,362.96",2,3,5,10,At Risk
2127,"15,245.00",134,3,"2,515.84",2,3,5,10,At Risk


## 7.2 Validate Segment Sizes

### 7.2.1 Customer Count per Segment

In [42]:
segment_counts = (
    rfm['segment']
    .value_counts()
    .reset_index(name='customer_count')
)

segment_counts['percentage'] = (
    segment_counts['customer_count']
    / len(rfm)
    * 100
).round(2)

display(segment_counts)

,segment,customer_count,percentage
0,Champions,957,22.06
1,At Risk,870,20.06
2,Loyal Customers,693,15.98
3,Potential Loyalists,633,14.59
4,Lost Customers,563,12.98
5,New / Developing Customers,537,12.38
6,Low Value Loyalists,85,1.96


### 7.2.2 Total Segment Customers

In [43]:
total_segment_customers = segment_counts['customer_count'].sum()
total_rfm_customers = rfm['CustomerID'].nunique()

print("Total segment customers :", total_segment_customers)
print("Total RFM customers     :", total_rfm_customers)
print("Difference              :", total_segment_customers - total_rfm_customers)
print("Customer count match    :", total_segment_customers == total_rfm_customers)

Total segment customers : 4338
Total RFM customers     : 4338
Difference              : 0
Customer count match    : True


### 7.2.3 Segment Percentage Validation

In [44]:
total_percentage = segment_counts['percentage'].sum()

print("Total segment percentage :", round(total_percentage, 2))
print(
    "Percentage match         :",
    abs(total_percentage - 100) < 0.01
)

Total segment percentage : 100.01
Percentage match         : True


### 7.2.4 Missing Segment

In [45]:
missing_segment = rfm['segment'].isna().sum()

print("Missing segment :", missing_segment)
print("Missing segment validation :", missing_segment == 0)

Missing segment : 0
Missing segment validation : True


### 7.2.5 Duplicate Customer Validation

In [46]:
duplicate_customers = (
    rfm['CustomerID']
    .duplicated()
    .sum()
)

print("Duplicate customers :", duplicate_customers)
print(
    "Duplicate validation :",
    duplicate_customers == 0
)

Duplicate customers : 0
Duplicate validation : True


### 7.2.6 Segment Coverage Validation

In [48]:
expected_segments = {
    'Champions',
    'Low Value Loyalists',
    'Loyal Customers',
    'Potential Loyalists',
    'At Risk',
    'Lost Customers',
    'New / Developing Customers'
}

actual_segments = set(rfm['segment'].dropna().unique())

unexpected_segments = actual_segments - expected_segments
missing_segments = expected_segments - actual_segments

print("Expected segments :", len(expected_segments))
print("Actual segments   :", len(actual_segments))

print("Unexpected segments :", unexpected_segments)
print("Missing segments    :", missing_segments)

print(
    "Segment coverage match :",
    actual_segments == expected_segments
)

Expected segments : 7
Actual segments   : 7
Unexpected segments : set()
Missing segments    : set()
Segment coverage match : True


## 7.3 Analyze Revenue Contribution

### 7.3.1 Revenue Summary by Segment

In [49]:
segment_revenue = (
    rfm.groupby('segment')
    .agg(
        customer_count=('CustomerID', 'nunique'),
        total_revenue=('monetary', 'sum')
    )
    .reset_index()
)

segment_revenue = segment_revenue.sort_values(
    'total_revenue',
    ascending=False
).reset_index(drop=True)

display(segment_revenue)

,segment,customer_count,total_revenue
0,Champions,957,"5,791,640.74"
1,Loyal Customers,693,"1,298,467.95"
2,At Risk,870,"594,183.34"
3,Potential Loyalists,633,"557,965.27"
4,New / Developing Customers,537,"325,576.66"
5,Lost Customers,563,"290,552.05"
6,Low Value Loyalists,85,"28,822.88"


### 7.3.2 Revenue Contribution %

In [50]:
total_customer_revenue = rfm['monetary'].sum()

segment_revenue['revenue_contribution_pct'] = (
    segment_revenue['total_revenue']
    / total_customer_revenue
    * 100
).round(2)

display(segment_revenue)

,segment,customer_count,total_revenue,revenue_contribution_pct
0,Champions,957,"5,791,640.74",65.17
1,Loyal Customers,693,"1,298,467.95",14.61
2,At Risk,870,"594,183.34",6.69
3,Potential Loyalists,633,"557,965.27",6.28
4,New / Developing Customers,537,"325,576.66",3.66
5,Lost Customers,563,"290,552.05",3.27
6,Low Value Loyalists,85,"28,822.88",0.32


### 7.3.3 Average Revenue per Customer

In [51]:
segment_revenue['average_revenue_per_customer'] = (
    segment_revenue['total_revenue']
    / segment_revenue['customer_count']
).round(2)

display(segment_revenue)

,segment,customer_count,total_revenue,revenue_contribution_pct,average_revenue_per_customer
0,Champions,957,"5,791,640.74",65.17,"6,051.87"
1,Loyal Customers,693,"1,298,467.95",14.61,"1,873.69"
2,At Risk,870,"594,183.34",6.69,682.97
3,Potential Loyalists,633,"557,965.27",6.28,881.46
4,New / Developing Customers,537,"325,576.66",3.66,606.29
5,Lost Customers,563,"290,552.05",3.27,516.08
6,Low Value Loyalists,85,"28,822.88",0.32,339.09


## 7.4 Build Customer-Level Dataset

In [58]:
customer_level = rfm[
    [
        'CustomerID',
        'recency',
        'frequency',
        'monetary',
        'R_score',
        'F_score',
        'M_score',
        'RFM_score',
        'segment'
    ]
].copy()

customer_level = customer_level.sort_values(
    'CustomerID'
).reset_index(drop=True)

display(customer_level.head(10))

,CustomerID,recency,frequency,monetary,R_score,F_score,M_score,RFM_score,segment
0,"12,346.00",326,1,"77,183.60",1,1,5,7,Lost Customers
1,"12,347.00",2,7,"4,310.00",5,5,5,15,Champions
2,"12,348.00",75,4,"1,797.24",2,4,4,10,Loyal Customers
3,"12,349.00",19,1,"1,757.55",4,1,4,9,Potential Loyalists
4,"12,350.00",310,1,334.40,1,1,2,4,Lost Customers
5,"12,352.00",36,8,"2,506.04",3,5,5,13,Loyal Customers
6,"12,353.00",204,1,89.00,1,1,1,3,Lost Customers
7,"12,354.00",232,1,"1,079.40",1,1,4,6,Lost Customers
8,"12,355.00",214,1,459.40,1,1,2,4,Lost Customers
9,"12,356.00",23,3,"2,811.43",4,3,5,12,Potential Loyalists


In [59]:
print("Rows    :", len(customer_level))
print("Columns :", len(customer_level.columns))

Rows    : 4338
Columns : 9


In [68]:
validation = {
    'customer_count': len(customer_level),
    'unique_customer_count': customer_level['CustomerID'].nunique(),
    'duplicate_customer_count': customer_level['CustomerID'].duplicated().sum(),
    'missing_value_count': customer_level.isnull().sum().sum(),
    'segment_count': customer_level['segment'].nunique(),
    'invalid_rfm_score_count': (
        ~customer_level['RFM_score'].between(3, 15)
    ).sum()
}

display(pd.Series(validation))

customer_count              4338
unique_customer_count       4338
duplicate_customer_count       0
missing_value_count            0
segment_count                  7
invalid_rfm_score_count        0
dtype: int64

In [69]:
customer_level.to_csv(
    '../data/processed/customer_rfm_segments.csv',
    index=False
)

In [70]:
import os

output_path = '../data/processed/customer_rfm_segments.csv'

print("File exists :", os.path.exists(output_path))

if os.path.exists(output_path):
    print("File size   :", f"{os.path.getsize(output_path) / 1024:.2f} KB")

File exists : True
File size   : 195.90 KB
